# 00 - Math Review For micrograd

这本 `course` 不是压缩版提纲。课堂 notebook 会先把直觉、例子、代码和图跑通，再进入最后的 Predict / Modify 检查。

学习顺序：先读这一页的主线，遇到代码就运行；最后再做底部的小检查。真正写作业时进入同目录的 `homework.ipynb`。

In [ ]:
from pathlib import Path
import sys, math, random, inspect


def find_repo_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / 'micrograd' / 'engine.py').exists():
            return q
    return p

ROOT = find_repo_root()
PROJECT_ROOT = ROOT
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from micrograd.engine import Value
from micrograd.nn import Neuron, Layer, MLP

try:
    import matplotlib.pyplot as plt
    MATPLOTLIB_AVAILABLE = True
except ModuleNotFoundError:
    plt = None
    MATPLOTLIB_AVAILABLE = False

try:
    import torch
except ModuleNotFoundError:
    torch = None


def near(a, b, eps=1e-6):
    return abs(a - b) < eps


def assert_close(actual, expected, name='value', eps=1e-9):
    assert abs(actual - expected) < eps, f'{name}: expected {expected}, got {actual}'


def todo_guard(values, message='请先填写 TODO，再运行检查。'):
    if any(v is None for v in values):
        print(message)
        return False
    return True

import importlib
import course.checks as course_checks
importlib.reload(course_checks)
qa_check = course_checks.qa_check


def show_svg(path):
    path = Path(path)
    try:
        from IPython.display import SVG, display
        display(SVG(filename=str(path)))
    except Exception:
        print('图已生成，但当前环境不能内嵌显示。请打开:', path.resolve())

print('project root:', ROOT)


## 怎么学这本

这本只解决一个问题：**为什么 `grad` 可以理解成变化率？**

建议用法：

1. 先读文字和公式，不急着理解画图代码的每一行
2. 看到图时，只问自己：横轴是什么，纵轴是什么，箭头代表什么
3. 读完后，能用自己的话解释“导数、偏导、梯度、链式法则”的区别

你可以暂时跳过：`matplotlib` 画图 API 的细节。那些只是为了辅助理解，不是 micrograd 的核心。


## 本节学习闭环

这一节按这个顺序学，不要跳：

```text
直觉 -> 小数字 -> 图像 -> 公式 -> 代码验证 -> 自测
```

你每看到一个公式，都问自己两件事：

```text
1. 它是在问哪个变量变一点，结果怎么变？
2. 如果代入小数字，我能不能算出同一个答案？
```

本节最后的过关标准不是会背定义，而是能把这句话说顺：

```text
反向传播就是把最终 loss 对每个中间变量的变化率，一层层用链式法则传回去。
```

## 关键词对照

| 词 | 先这样理解 |
|---|---|
| 导数 | 一元函数里，某一点的斜率 |
| 偏导 | 多元函数里，只让一个变量变时的斜率 |
| 梯度 | 所有偏导组成的向量 |
| 链式法则 | 中间变量一层层传递影响，变化率沿路径相乘 |
| loss | 模型当前错得有多离谱 |
| grad | 某个变量变一点，最终 loss 会怎么变 |


In [ ]:
try:
    import numpy as np
    NUMPY_AVAILABLE = True
except Exception as exc:
    NUMPY_AVAILABLE = False
    print('numpy unavailable:', exc)

try:
    import matplotlib.pyplot as plt
    MATPLOTLIB_AVAILABLE = True
except Exception as exc:
    MATPLOTLIB_AVAILABLE = False
    print('matplotlib unavailable:', exc)

if not (NUMPY_AVAILABLE and MATPLOTLIB_AVAILABLE):
    print('本节画图会自动跳过；公式和手算练习仍然可以继续。')

## 1. 一元函数：导数就是斜率

如果只有一个输入：

$$
y = f(x)
$$

那导数就是曲线在某一点的斜率：

$$
\frac{dy}{dx} = f'(x)
$$

例如：

$$
y = x^2
$$

它的导数是：

$$
\frac{dy}{dx} = 2x
$$

当 $x=2$ 时，斜率是：

$$
f'(2)=4
$$


In [ ]:
if NUMPY_AVAILABLE and MATPLOTLIB_AVAILABLE:
    x = np.linspace(-1, 4, 200)
    y = x ** 2

    x0 = 2.0
    y0 = x0 ** 2
    slope = 2 * x0

    tangent_x = np.linspace(0.5, 3.5, 50)
    tangent_y = y0 + slope * (tangent_x - x0)

    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    ax.plot(x, y, label='y = x^2')
    ax.plot(tangent_x, tangent_y, '--', label="tangent at x=2, slope=4")
    ax.scatter([x0], [y0], color='crimson', zorder=3)
    ax.annotate('(2, 4)', (x0, y0), xytext=(2.15, 3.2), color='crimson')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title('Derivative: slope of a curve at one point')
    ax.grid(True, alpha=0.25)
    ax.legend()
    plt.show()
else:
    print('cell 10：numpy/matplotlib 不可用，跳过画图。')

### Mini Lab: 先预测斜率，再运行

不要先运行。先在脑子里预测：

```text
y = x^2
x = 0, 1, 2, 3 时，斜率分别是多少？
```

然后运行下面的 cell。

In [ ]:
def slope_of_square_at(x_value):
    return 2 * x_value

for x_value in [0, 1, 2, 3]:
    print(f'x={x_value}, slope={slope_of_square_at(x_value)}')

## 2. 二元函数：偏导是某个方向上的斜率

micrograd 例子里有两个输入：

$$
L = f(a,b) = 2(ab + a)
$$

这时候不能只说“一个斜率”，因为有两个方向可以动：

- 固定 $b$，只让 $a$ 变，得到 $a$ 方向的斜率
- 固定 $a$，只让 $b$ 变，得到 $b$ 方向的斜率

这两个斜率叫偏导：

$$
\frac{\partial L}{\partial a}, \quad \frac{\partial L}{\partial b}
$$


## 3. 手算这个例子的偏导

原式：

$$
L = 2(ab + a)
$$

对 $a$ 求偏导时，把 $b$ 当常数：

$$
\frac{\partial L}{\partial a}
= 2\left(\frac{\partial(ab)}{\partial a} + \frac{\partial a}{\partial a}\right)
= 2(b + 1)
$$

对 $b$ 求偏导时，把 $a$ 当常数：

$$
\frac{\partial L}{\partial b}
= 2 \frac{\partial(ab)}{\partial b}
= 2a
$$

代入 $a=2, b=3$：

$$
\frac{\partial L}{\partial a} = 2(3+1)=8
$$

$$
\frac{\partial L}{\partial b} = 2 \times 2=4
$$


## 4. 梯度：所有偏导组成的向量

因为 $L$ 有两个输入 $a,b$，所以它的梯度是一个向量：

$$
\nabla L(a,b)
= \left[\frac{\partial L}{\partial a}, \frac{\partial L}{\partial b}\right]
$$

在 $(a,b)=(2,3)$ 这个点：

$$
\nabla L(2,3) = [8,4]
$$

直观理解：

- $a$ 稍微增大一点，$L$ 大约按 8 倍速度变化
- $b$ 稍微增大一点，$L$ 大约按 4 倍速度变化
- 梯度箭头指向 $L$ 上升最快的方向


In [ ]:
if NUMPY_AVAILABLE and MATPLOTLIB_AVAILABLE:
    A, B = np.meshgrid(np.linspace(-1, 5, 140), np.linspace(-1, 5, 140))
    Z = 2 * (A * B + A)

    point_a, point_b = 2.0, 3.0
    grad_a, grad_b = 2 * (point_b + 1), 2 * point_a

    fig, ax = plt.subplots(figsize=(6.5, 5.2))
    contours = ax.contour(A, B, Z, levels=16, cmap='viridis')
    ax.clabel(contours, inline=True, fontsize=8)

    ax.scatter([point_a], [point_b], color='crimson', s=60, zorder=3)
    ax.quiver(
        [point_a], [point_b], [grad_a], [grad_b],
        angles='xy', scale_units='xy', scale=8,
        color='crimson', width=0.008,
    )

    ax.annotate('point (a=2, b=3)', (point_a, point_b), xytext=(2.2, 2.65), color='crimson')
    ax.annotate('grad = [8, 4]', (point_a + 1.0, point_b + 0.5), color='crimson')
    ax.set_xlabel('a')
    ax.set_ylabel('b')
    ax.set_title('L = 2(ab + a): contour lines and gradient')
    ax.grid(True, alpha=0.25)
    plt.show()
else:
    print('cell 16：numpy/matplotlib 不可用，跳过画图。')

## Chain Rule In One Variable

链式法则先用一句话理解：

```text
如果结果不是直接由 x 算出来，而是经过中间变量一层层算出来，
最终变化率 = 每一层局部变化率相乘。
```

比如：

$$
y = (3x + 1)^2
$$

可以拆成两步：

$$
u = 3x + 1
$$

$$
y = u^2
$$

这时：

$$
\frac{dy}{dx}
=
\frac{dy}{du}
\cdot
\frac{du}{dx}
$$

分别算局部导数：

$$
\frac{dy}{du}=2u
$$

$$
\frac{du}{dx}=3
$$

所以：

$$
\frac{dy}{dx}=2u \cdot 3
$$

当 $x=2$ 时，$u=3\times2+1=7$，所以：

$$
\frac{dy}{dx}=2\times7\times3=42
$$

直觉是：

```text
x 变一点 -> u 按 3 倍变化 -> y 按 2u 倍变化
所以 x 对 y 的最终影响是 3 * 2u
```


In [ ]:
x0 = 2.0
u0 = 3 * x0 + 1
y0 = u0 ** 2

du_dx = 3
dy_du = 2 * u0
dy_dx = dy_du * du_dx

print('x =', x0)
print('u = 3x + 1 =', u0)
print('y = u^2 =', y0)
print('du/dx =', du_dx)
print('dy/du =', dy_du)
print('dy/dx = dy/du * du/dx =', dy_dx)

## Chain Rule In micrograd's Example

micrograd 里的例子：

$$
L = 2(ab+a)
$$

拆成中间变量：

$$
c = ab
$$

$$
d = c + a
$$

$$
L = 2d
$$

如果只看路径 $a \to c \to d \to L$，链式法则就是：

$$
\frac{\partial L}{\partial a}
=
\frac{\partial L}{\partial d}
\cdot
\frac{\partial d}{\partial c}
\cdot
\frac{\partial c}{\partial a}
$$

这一条路径的贡献是：

$$
2 \times 1 \times b
$$

当 $b=3$ 时：

$$
2 \times 1 \times 3 = 6
$$

但是 $a$ 还有另一条路径 $a \to d \to L$：

$$
\frac{\partial L}{\partial d}
\cdot
\frac{\partial d}{\partial a}
= 2 \times 1 = 2
$$

所以两条路径相加：

$$
\frac{\partial L}{\partial a}=6+2=8
$$

这就是：

```text
路径上局部导数相乘；多条路径回到同一个变量时相加。
```


## 5. 链式法则：反向传播的核心

micrograd 不会直接看见这个完整公式：

$$
L = 2(ab+a)
$$

它看到的是一步一步的小计算：

$$
c = ab
$$

$$
d = c + a
$$

$$
L = 2d
$$

计算图是：

```text
a ----\
       * ---> c ----\
b ----/              + ---> d ---> *2 ---> L
             a ----/
```

反向传播做的事，就是从右往左乘局部斜率。


In [ ]:
if NUMPY_AVAILABLE and MATPLOTLIB_AVAILABLE:
    fig, ax = plt.subplots(figsize=(9, 3.6))
    ax.set_axis_off()

    nodes = {
        'a': (0.0, 1.2, 'a\n2'),
        'b': (0.0, 0.0, 'b\n3'),
        '*1': (1.8, 0.6, '*'),
        'c': (3.2, 0.6, 'c=a*b\n6'),
        '+': (4.8, 1.0, '+'),
        'd': (6.2, 1.0, 'd=c+a\n8'),
        '*2': (7.8, 1.0, '*2'),
        'L': (9.2, 1.0, 'L\n16'),
    }

    edges = [('a', '*1'), ('b', '*1'), ('*1', 'c'), ('c', '+'), ('a', '+'), ('+', 'd'), ('d', '*2'), ('*2', 'L')]

    for start, end in edges:
        x1, y1, _ = nodes[start]
        x2, y2, _ = nodes[end]
        ax.annotate('', xy=(x2 - 0.25, y2), xytext=(x1 + 0.25, y1), arrowprops=dict(arrowstyle='->', lw=1.7, color='#444'))

    for name, (x, y, text) in nodes.items():
        is_op = name in {'*1', '+', '*2'}
        ax.text(
            x, y, text,
            ha='center', va='center', fontsize=11,
            bbox=dict(boxstyle='round,pad=0.35', fc='#fff4cc' if is_op else '#e8f3ff', ec='#666', lw=1.2),
        )

    ax.text(4.6, -0.55, 'forward: compute data left -> right', ha='center', fontsize=11)
    ax.text(4.6, -0.85, 'backward: send grad right -> left', ha='center', fontsize=11)
    ax.set_xlim(-0.7, 9.9)
    ax.set_ylim(-1.1, 1.8)
    plt.show()
else:
    print('cell 21：numpy/matplotlib 不可用，跳过画图。')

## 6. 用链式法则算回 a 和 b

从最后开始：

$$
L = 2d \Rightarrow \frac{\partial L}{\partial d}=2
$$

因为：

$$
d = c + a
$$

所以：

$$
\frac{\partial d}{\partial c}=1, \quad \frac{\partial d}{\partial a}=1
$$

因为：

$$
c = ab
$$

所以：

$$
\frac{\partial c}{\partial a}=b=3, \quad \frac{\partial c}{\partial b}=a=2
$$

现在把路径乘起来。$a$ 有两条路径影响 $L$：

路径 1：$a \to d \to L$

$$
2 \times 1 = 2
$$

路径 2：$a \to c \to d \to L$

$$
2 \times 1 \times 3 = 6
$$

两条路径相加：

$$
\frac{\partial L}{\partial a}=2+6=8
$$

$b$ 只有一条路径：$b \to c \to d \to L$

$$
\frac{\partial L}{\partial b}=2 \times 1 \times 2=4
$$

这就是 micrograd 的 `backward()` 在代码里做的事。


## 7. 你自己手算一下

换一个例子：

$$
z = 3(xy + y)
$$

当 $x=4, y=-2$ 时：

$$
\frac{\partial z}{\partial x} = ?
$$

$$
\frac{\partial z}{\partial y} = ?
$$

先手算，再去 `01_gradient_practice.ipynb` 里运行对应练习 cell 看答案。


## 8. 过关小测：不用 micrograd

先自己在纸上算，再运行下面的 cell。

对于：

$$
L = 2(ab+a)
$$

你应该能写出：

$$
\frac{\partial L}{\partial a}=2(b+1), \quad \frac{\partial L}{\partial b}=2a
$$

然后代入不同的 $a,b$。

In [ ]:
def hand_grad_for_L(a, b):
    dL_da = 2 * (b + 1)
    dL_db = 2 * a
    return dL_da, dL_db

checks = [
    ((2, 3), (8, 4)),
    ((4, -2), (-2, 8)),
    ((0, 5), (12, 0)),
]

for (a_value, b_value), expected in checks:
    actual = hand_grad_for_L(a_value, b_value)
    print(f'a={a_value:>4}, b={b_value:>4} -> grad={actual}')
    assert actual == expected

print('OK: 你已经能手算这个核心例子的梯度。')

## What To Remember

- 跑通主线。
- 说清楚本节的输入、输出、梯度或训练含义。
- 完成底部课堂检查后再进入 homework。

## 课堂检查：先预测，再改一行

这一段保留 `course` 的隐藏检查。你应该先自己填，再展开提示或答案。

## Predict - 先猜斜率

`y = 2x + 1` 在 `x=3` 附近的斜率是多少？

In [ ]:
# 填写说明：
# - 填一个数字：先不用代码，预测局部斜率。
# - 填完后运行本 cell，看 qa_check 的提示。

predict_slope = None



qa_check('qa_check_class_00_predict', globals())

<details><summary>Show / Hide 本题提示</summary>

- 先用小数字画路径：一元题看斜率，多元题固定另一个变量，链式题把路径贡献相乘。

</details>

<details><summary>Show / Hide 本题答案</summary>


参考答案（先自己做，卡住再展开）：

- `predict_slope` 填 `2`。

</details>

## Run - 用很小的 h 看局部变化率

In [ ]:
def f(x):
    return 2*x + 1

x0 = 3.0
h = 0.0001
print((f(x0+h) - f(x0)) / h)

## 拆开看 - 多元函数要固定别的变量

`L = 2(ab+a)`，在 `a=2,b=3` 时：

```text
dL/da = 2(b+1) = 8
dL/db = 2a = 4
```

In [ ]:
a = 2
b = 3
paths = [
    ('a -> ab -> ab+a -> L', 'b', '2', b*2),
    ('a -> a  -> ab+a -> L', '1', '2', 1*2),
    ('b -> ab -> ab+a -> L', 'a', '2', a*2),
]
for path, local, upstream, contrib in paths:
    print(f'{path:24s} local={local:>2s} upstream={upstream:>2s} contribution={contrib}')
print('dL/da = 6 + 2 = 8, dL/db = 4')

## Modify - 换一个链式法则

In [ ]:
# 填写说明：
# - 填一个数字：对 y=(3x+1)^2 在 x=2 处手算 dy/dx。
# - 填完后运行本 cell，看 qa_check 的提示。

# y=(3x+1)^2, x=2
student_dy_dx = None



qa_check('qa_check_class_00_modify', globals())

<details><summary>Show / Hide 本题提示</summary>

- 先用小数字画路径：一元题看斜率，多元题固定另一个变量，链式题把路径贡献相乘。

</details>

<details><summary>Show / Hide 本题答案</summary>


参考答案（先自己做，卡住再展开）：

- `student_dy_dx` 填 `42`。

</details>

## 接回 micrograd

以后看到 `some_value.grad`，先翻译成一句话：最终 `L` 对 `some_value` 的偏导。